In [ ]:
# Import modules
import paramiko
import os
from stat import S_ISDIR
import pandas as pd
import dotenv

In [ ]:
# Provide connection details in separate .env file for security
dotenv.load_dotenv()
hostname = os.getenv('HOSTNAME')  # Replace with your server's IP or hostname
user = os.getenv('USER')    # Replace with your SSH username    
password = os.getenv('PASSWORD')  # SSH password from .env file

PROJECT_DIRECTORY = '/mnt/data_raid/feierabend/AVL_FIRE/PEMWE/PEMStar_2'    # Project directory on the remote server
MODEL_NAME = 'PEMStar_BekaertPTL'                   # AVL FIRE model name
CASE_SET_NAME = 'PolCurve_Bek~rtPTL_Update'        # Case set name within the model
CASE_NAME = None                                    # Set to None to search all cases, or specify a case name like 'Case_1'

# Or, if you want to search for files with specific names
# TARGET_FILENAMES = ['file1.txt', 'report.log']


In [ ]:
# --- SSH Client Initialization ---
ssh_client = paramiko.SSHClient()
# Automatically add the server's host key. For production, it's better to manage known_hosts explicitly.
ssh_client.set_missing_host_key_policy(paramiko.AutoAddPolicy())

try:
    # Connect to the SSH server
    ssh_client.connect(hostname=hostname, username=user, password=password)
    print(f"Connected to {hostname} using password.")

    # Open an SFTP client
    sftp_client = ssh_client.open_sftp()
    print(f"Opened SFTP session.")
except paramiko.AuthenticationException:
    print("Authentication failed. Check your username, password, or private key.")
except paramiko.SSHException as e:
    print(f"Could not establish SSH connection: {e}")  

In [ ]:
# Define function for retrieving remote data files from AVL FIRE model cases
def retrieve_avl_fire_data_paths(sftp_client, project_directory, model_name, case_set_name, data_directory, file_extension, case_name=None):
    simulation_project_path = f"{project_directory}/simulation/project/"
    if case_name is None:
        print("case_name is None, searching for all cases in the specified model and case set...")
        data_paths = []
        for entry in sftp_client.listdir_attr(simulation_project_path):
            if S_ISDIR(entry.st_mode) and f"{model_name}.{case_set_name}." in entry.filename:
                case_path = f"{simulation_project_path}{entry.filename}"
                data_path = f"{case_path}/{data_directory}/{model_name}{file_extension}"
                data_paths.append(data_path)
                print(f"Found case: {entry.filename}, data path: {data_path}")
    else:
        case_set_path = f"{simulation_project_path}{model_name}.{case_set_name}"
        data_path = f"{case_set_path}.{case_name}/{data_directory}/{model_name}{file_extension}"
        print(f"Using specified case_name: {case_name}, data path: {data_path}")
        data_paths = [f"{data_path}"]
    return data_paths
    

# Retrieve Simulation Input Data
Get input from ".asix" file

In [ ]:
input_data_paths = retrieve_avl_fire_data_paths(
    sftp_client=sftp_client,
    project_directory=PROJECT_DIRECTORY,
    model_name=MODEL_NAME,
    case_set_name=CASE_SET_NAME,
    data_directory='input',
    file_extension='.asix'
)

In [ ]:
from src.asix_parser import parse_asix
input_data_dicts_list = []
for data_path in input_data_paths:
    with sftp_client.open(data_path, 'r') as data_file:
        # data = remote_file.read()
        data = parse_asix(data_file, always_list=False, keep_all_attributes=True, cast_values=True)
        input_data_dicts_list.append(data)
    # data_dicts
input_data_dicts_list

In [ ]:
input_data = input_data_dicts_list[0]

In [ ]:
# from src.utils import _json_default
# with open('testing_output_v3.json', 'w', encoding='utf-8') as f:
#     json.dump(data_dicts, f, default=_json_default, indent=2)

# Retrieve Simulation Input Data
Get input from ".asix" file

In [ ]:
results_2d_data_paths = retrieve_avl_fire_data_paths(
    sftp_client=sftp_client,
    project_directory=PROJECT_DIRECTORY,
    model_name=MODEL_NAME,
    case_set_name=CASE_SET_NAME,
    data_directory='results',
    file_extension='.csv'
)
print(results_2d_data_paths)

In [ ]:
data_path = results_2d_data_paths[0]
result_2d_result_list = []
for data_path in results_2d_data_paths:
    with sftp_client.open(data_path, 'r') as data_file:
        df = pd.read_csv(data_file, header=[1,2], sep=';')  # Adjust separator if needed
        result_2d_result_list.append(df)
result_2d = result_2d_result_list[0]

In [ ]:
results_monitoring_data_paths = retrieve_avl_fire_data_paths(
    sftp_client=sftp_client,
    project_directory=PROJECT_DIRECTORY,
    model_name=MODEL_NAME,
    case_set_name=CASE_SET_NAME,
    data_directory='results',
    file_extension='_flc.csv'
)
print(results_monitoring_data_paths)

In [ ]:
data_path = results_monitoring_data_paths[0]
result_monitoring_result_list = []
for data_path in results_monitoring_data_paths:
    with sftp_client.open(data_path, 'r') as data_file:
        df = pd.read_csv(data_file, header=[1,2], sep=';')  # Adjust separator if needed
        result_monitoring_result_list.append(df)
result_monitoring_result_list[0].head()

In [ ]:
# Close the connections
if 'sftp_client' in locals() and sftp_client:
    sftp_client.close()
    print("SFTP session closed.")
if 'ssh_client' in locals() and ssh_client:
    ssh_client.close()
    print("SSH connection closed.")

In [ ]:
import importlib
import src.firem_name_parser_integration as firem_parser
importlib.reload(firem_parser)
from src.firem_name_parser_integration import (
    normalize_2d_results_columns,
    rename_2d_results_columns,
    normalize_case_bundle,
    load_yaml_from_github
)
# from src.firem_name_parser_integration import load_yaml_from_github

rules_path = load_yaml_from_github()

# Single case
mapping_df = normalize_2d_results_columns(result_2d, input_data, rules_path)
df_2d_renamed, rename_map = rename_2d_results_columns(result_2d, input_data, rules_path)

# # Multiple cases
# cases = {
#     "case_001": {"df_2d": df_case_001, "asix_dict": asix_case_001},
#     "case_002": {"df_2d": df_case_002, "asix_dict": asix_case_002},
# }
# normalized = normalize_case_bundle(cases, rules_path)

# mapping_df_case_001 = normalized["case_001"]["mapping_df"]
# df_case_001_renamed = normalized["case_001"]["df_renamed"]

In [ ]:
df_2D_renamed_2 = df_2d_renamed.rename(columns=rename_map)
df_2D_renamed_3 = df_2D_renamed_2.reset_index()
df_2D_renamed_3.columns = df_2D_renamed_3.columns.map(', '.join)


In [ ]:
df_4 = df_2D_renamed_2.copy()
new_cols = df_4.columns.droplevel(1)
df_4.columns = new_cols
df_4.columns

In [ ]:
df_5 = df_2d_renamed.copy()
df_5.columns = df_5.columns.get_level_values(0)
df_5.columns

In [ ]:
df_2D_renamed_3.columns

In [ ]:
list(rename_map.keys())[0]

In [ ]:
mapping_df

In [ ]:
df_2d_renamed